# 🌿 Plant Disease Detection (PDD) — ImageDataGenerator Version
### CNN Classifier: Early Blight | Late Blight | Healthy

This version uses the traditional `ImageDataGenerator` for data loading and augmentation.

---

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras.preprocessing.image import ImageDataGenerator
from keras import layers, models

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)

## 2. Configuration

In [ ]:
DATASET_DIR = "./training/PlantVillage"
IMAGE_SIZE  = 256
BATCH_SIZE  = 32
EPOCHS      = 30
NUM_CLASSES = 3

print("Dataset path:", os.path.abspath(DATASET_DIR))

## 3. Set up ImageDataGenerator

In [ ]:
# Training Generator with Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2 # Using 20% of data for validation
)

# Validation Generator (only rescale)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Flow from Directory
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='training',
    seed=42
)

validation_generator = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='validation',
    seed=42
)

class_names = list(train_generator.class_indices.keys())
print("Class Names:", class_names)

## 4. Build the Model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name="PDD_IDG_Model")

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Train the Model

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

## 6. Save the Model

In [ ]:
os.makedirs("./saved_models", exist_ok=True)
model.save("./saved_models/pdd_model_idg.keras")
print("✅ Model saved successfully!")